# Breaking the Feedback Trap: Massive Cross-Domain Evaluation (5 Datasets)
Notebook này tự động quét TẤT CẢ các dataset được add vào (5 datasets), lặp qua từng cái một, tiến hành train Baseline và Proposed model độc lập trên từng domain, và lưu toàn bộ kết quả để phục vụ cho Rebuttal.

## Cell 1: Setup & Environment

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import os, cv2, glob, time, random, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, seaborn as sns
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
os.makedirs("outputs", exist_ok=True)

## Cell 2: Auto-detect ALL Datasets

In [ ]:
# Tự động tìm tất cả các thư mục chứa cả 'images' và 'masks' (không bị ngắt ở dataset đầu tiên)
DATASETS = []
for root, dirs, _ in os.walk("/kaggle/input"):
    dirs_lower = [d.lower() for d in dirs]
    if "images" in dirs_lower and "masks" in dirs_lower:
        DATASETS.append(root)

# Loại bỏ các folder lồng nhau trùng lặp nếu có
DATASETS = list(set(DATASETS))

print(f"✅ Found {len(DATASETS)} valid datasets in /kaggle/input:")
for idx, ds in enumerate(DATASETS):
    print(f"  [{idx+1}] {ds}")

if not DATASETS:
    print("⚠️ No real datasets found! Using 'synthetic' fallback.")
    DATASETS = ["synthetic"]

## Cell 3: Data Loader & Model Architecture

In [ ]:
class MedicalSegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.is_synthetic = (image_paths == "synthetic")

    def __len__(self):
        return 100 if self.is_synthetic else len(self.image_paths)

    def __getitem__(self, idx):
        if self.is_synthetic:
            image = np.random.randint(50, 200, (256, 256, 3), dtype=np.uint8)
            mask = np.zeros((256, 256), dtype=np.float32)
            cv2.circle(mask, (128 + random.randint(-20,20), 128 + random.randint(-20,20)), random.randint(30, 60), 1, -1)
        else:
            image = cv2.imread(self.image_paths[idx])
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            if mask is None: mask = np.zeros((256, 256), dtype=np.uint8)
            mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        
        return image, mask.unsqueeze(0)

train_transform = A.Compose([
    A.Resize(256, 256), A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(256, 256), A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()
])

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth
    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        TP = (inputs * targets).sum()
        FP = ((1 - targets) * inputs).sum()
        FN = (targets * (1 - inputs)).sum()
        return 1 - ((TP + self.smooth) / (TP + self.alpha * FN + self.beta * FP + self.smooth))

class MiniRecurrentNet(nn.Module):
    def __init__(self, mode='baseline'):
        super().__init__()
        self.mode = mode 
        self.enc = nn.Sequential(nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.dec = nn.Conv2d(32, 1, 3, padding=1)
        
    def forward(self, x, iters=2):
        B, C, H, W = x.shape
        m_prev = torch.zeros(B, 1, H, W, device=x.device)
        for t in range(iters):
            inp = torch.cat([x, m_prev], dim=1)
            feat = self.enc(inp)
            f_guidance = torch.sigmoid(self.dec(feat)) 
            
            if self.mode == 'baseline': m_prev = f_guidance 
            elif self.mode == 'proposed': m_prev = 1 - (1 - f_guidance.detach()) * (1 - m_prev)
        return m_prev 


## Cell 4: Training & Evaluation Functions

In [ ]:
def calc_metrics(preds, masks, thresh=0.5):
    preds_bin = (preds > thresh).float()
    masks = masks.float()
    TP = (preds_bin * masks).sum().item()
    FP = (preds_bin * (1 - masks)).sum().item()
    FN = ((1 - preds_bin) * masks).sum().item()
    TN = ((1 - preds_bin) * (1 - masks)).sum().item()
    dice = (2 * TP) / (2 * TP + FP + FN + 1e-6)
    fpr = FP / (FP + TN + 1e-6)
    return dice, fpr

def train_model(model, train_loader, val_loader, optimizer, criterion, name="Model", num_epochs=5, ds_name=""):
    history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_fpr': []}
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss, val_dice, val_fpr = 0.0, 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                out = model(imgs)
                loss = criterion(out, masks)
                dice, fpr = calc_metrics(out, masks)
                val_loss += loss.item()
                val_dice += dice
                val_fpr += fpr
                
        history['train_loss'].append(train_loss / len(train_loader))
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_dice'].append(val_dice / (len(val_loader) or 1))
        history['val_fpr'].append(val_fpr / (len(val_loader) or 1))
        print(f"  [{name}] Ep {epoch+1}/{num_epochs} | Val Dice: {history['val_dice'][-1]:.4f} | Val FPR: {history['val_fpr'][-1]:.4f}")
    
    pd.DataFrame(history).to_csv(f"outputs/{ds_name}_{name}_history.csv", index=False)
    torch.save(model.state_dict(), f"outputs/{ds_name}_{name}.pth")
    return history

def plot_and_visualize(h_base, h_prop, model_b, model_p, val_loader, ds_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    epochs = range(1, len(h_base['val_dice']) + 1)
    ax1.plot(epochs, h_base['val_dice'], 'r--', label='Baseline Dice')
    ax1.plot(epochs, h_prop['val_dice'], 'g-', label='Proposed Dice (Ours)')
    ax1.set_title(f'[{ds_name}] Validation Dice')
    ax1.legend()
    
    ax2.plot(epochs, h_base['val_fpr'], 'r--', label='Baseline FPR (Trap)')
    ax2.plot(epochs, h_prop['val_fpr'], 'g-', label='Proposed FPR (Firewall)')
    ax2.set_title(f'[{ds_name}] Validation FPR (Lower is Better)')
    ax2.legend()
    plt.savefig(f"outputs/{ds_name}_learning_curves.png")
    plt.close()
    
    model_b.eval(); model_p.eval()
    try:
        imgs, masks = next(iter(val_loader))
    except StopIteration:
        return
        
    with torch.no_grad():
        out_b = (model_b(imgs.to(device)).cpu() > 0.5).float()
        out_p = (model_p(imgs.to(device)).cpu() > 0.5).float()
    
    img_np = imgs[0].permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    gt = masks[0, 0].numpy()
    
    def get_overlay(img, truth, pred):
        overlay = img.copy()
        tp = (truth == 1) & (pred == 1)
        fp = (truth == 0) & (pred == 1)
        overlay[tp] = overlay[tp] * 0.3 + np.array([0, 1, 0]) * 0.7 
        overlay[fp] = overlay[fp] * 0.3 + np.array([1, 0, 0]) * 0.7 
        return np.clip(overlay, 0, 1)
    
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    axs[0].imshow(img_np); axs[0].set_title(f"[{ds_name}] Input")
    axs[1].imshow(gt, cmap='gray'); axs[1].set_title("Ground Truth")
    axs[2].imshow(get_overlay(img_np, gt, out_b[0,0].numpy())); axs[2].set_title("Baseline (Red = Trap)")
    axs[3].imshow(get_overlay(img_np, gt, out_p[0,0].numpy())); axs[3].set_title("Detached Soft-OR")
    for ax in axs: ax.axis('off')
    plt.savefig(f"outputs/{ds_name}_trap_visualization.png")
    plt.close()


## Cell 5: Automated Massive Cross-Domain Runner (Looping all 5 Datasets)

In [ ]:
for ds_path in DATASETS:
    ds_name = os.path.basename(ds_path) if ds_path != "synthetic" else "Synthetic"
    print(f"\n{'='*60}\n🚀 STARTING EXPERIMENT ON DATASET: {ds_name}\n{'='*60}")
    
    if ds_path == "synthetic":
        t_dataset = MedicalSegmentationDataset("synthetic", "synthetic", transform=train_transform)
        v_dataset = MedicalSegmentationDataset("synthetic", "synthetic", transform=val_transform)
    else:
        # Hỗ trợ cả file thường và file trong subfolders
        all_imgs = sorted(glob.glob(os.path.join(ds_path, 'images', '**', '*.*'), recursive=True))
        all_masks = sorted(glob.glob(os.path.join(ds_path, 'masks', '**', '*.*'), recursive=True))
        
        # Lọc ảnh rác
        all_imgs = [p for p in all_imgs if p.endswith(('.jpg', '.png', '.jpeg'))]
        all_masks = [p for p in all_masks if p.endswith(('.jpg', '.png', '.jpeg'))]
        
        split_idx = int(len(all_imgs) * 0.8)
        t_dataset = MedicalSegmentationDataset(all_imgs[:split_idx], all_masks[:split_idx], transform=train_transform)
        v_dataset = MedicalSegmentationDataset(all_imgs[split_idx:], all_masks[split_idx:], transform=val_transform)
    
    if len(t_dataset) == 0:
        print(f"⚠️ Skipped {ds_name}: No valid images found.")
        continue
        
    t_loader = DataLoader(t_dataset, batch_size=8, shuffle=True)
    v_loader = DataLoader(v_dataset, batch_size=8, shuffle=False)
    print(f"Dataset Split -> Train: {len(t_dataset)}, Val: {len(v_dataset)}")
    
    # Khởi tạo mô hình mới tinh cho từng dataset (tránh data leakage)
    model_b = MiniRecurrentNet(mode='baseline').to(device)
    model_p = MiniRecurrentNet(mode='proposed').to(device)
    criterion = TverskyLoss(alpha=0.3, beta=0.7)
    opt_b = torch.optim.AdamW(model_b.parameters(), lr=1e-3)
    opt_p = torch.optim.AdamW(model_p.parameters(), lr=1e-3)
    
    print(f"\n--- Training Baseline (Hard Feedback) on {ds_name} ---")
    h_base = train_model(model_b, t_loader, v_loader, opt_b, criterion, name="Baseline", num_epochs=5, ds_name=ds_name)
    
    print(f"\n--- Training Proposed (Detached Soft-OR) on {ds_name} ---")
    h_prop = train_model(model_p, t_loader, v_loader, opt_p, criterion, name="Proposed", num_epochs=5, ds_name=ds_name)
    
    print(f"\nSaving curves and visualizations for {ds_name}...")
    plot_and_visualize(h_base, h_prop, model_b, model_p, v_loader, ds_name)
    print(f"✅ Finished {ds_name}. Results saved to /outputs/")
    
print(f"\n{'='*60}\n🎉 ALL CROSS-DOMAIN EXPERIMENTS COMPLETED SUCCESSFULLY!\n{'='*60}")